In [1]:
import os
import numpy as np
import napari
import zipfile
from pathlib import Path

In [13]:
# --- Paths ---
sample_no = 16
base_dir = Path(os.getcwd()) / f"{sample_no}"
sample_folder = base_dir
raw_npz_file = sample_folder / "raw.npz"
raw_extract_dir = sample_folder / "raw_extracted_npz"

# --- Extract and memory-map raw image ---
raw_npy_path = raw_extract_dir / "arr_0.npy"
if not raw_npy_path.exists():
    raw_extract_dir.mkdir(exist_ok=True)
    with np.load(raw_npz_file) as raw_npz:
        raw_array = raw_npz['arr_0'] if 'arr_0' in raw_npz else list(raw_npz.values())[0]
        np.save(raw_npy_path, raw_array)

raw_data = np.load(raw_npy_path, mmap_mode='r')

# --- Load all mask folders ---
folders = sorted(base_dir.glob(f"Sample{sample_no}_thresh_adj_mult_*"))
mask_layers = []

for folder in folders:
    mult = folder.name.split("_")[-1]
    mask_npz_path = folder / "results_binary_masks.npz"
    mask_extract_dir = folder / "extracted_mask"
    mask_npy_path = mask_extract_dir / "mask.npy"

    if not mask_npz_path.exists():
        print(f"Skipping missing: {mask_npz_path}")
        continue

    if not mask_npy_path.exists():
        mask_extract_dir.mkdir(exist_ok=True)
        with np.load(mask_npz_path) as mask_npz:
            mask_array = mask_npz['arr_0'] if 'arr_0' in mask_npz else list(mask_npz.values())[0]
            np.save(mask_npy_path, mask_array)

    mask = np.load(mask_npy_path, mmap_mode='r')
    mask_layers.append((f"mask_mult_{mult}", mask))

# --- View in Napari ---
with napari.gui_qt():
    viewer = napari.Viewer()
    viewer.add_image(raw_data, name="Raw", blending="additive", colormap="gray")

    for name, mask in mask_layers:
        viewer.add_labels(mask, name=name, opacity=0.5, visible=False)

/home/volkan/micromamba/envs/filopodia_roi_selector/lib/python3.10/contextlib.py:135: FutureWarning: 
The 'gui_qt()' context manager is deprecated.
If you are running napari from a script, please use 'napari.run()' as follows:

    import napari

    viewer = napari.Viewer()  # no prior setup needed
    # other code using the viewer...
    napari.run()

In IPython or Jupyter, 'napari.run()' is not necessary. napari will automatically
start an interactive event loop for you: 

    import napari
    viewer = napari.Viewer()  # that's it!

  return next(self.gen)
/home/volkan/micromamba/envs/filopodia_roi_selector/lib/python3.10/site-packages/napari/_qt/qt_event_loop.py:338: FutureWarning: `QApplication` instance access through `get_app` is deprecated and will be removed in 0.6.0.
Please use `get_qapp` instead.

  app = get_app()
